# 🎮 Gamer Science · Wizards 101 · Applied Physics · Quantum Info

> **Every domain is a game. Every equation is a spell. Every tensor is a weapon.**

| § | Domain | Boss Spell |
|---|---|---|
| §1 | ⚡ EM Circuits | Maxwell → RLC → S-params → transmission lines |
| §2 | 🔐 Cryptography | RSA · AES S-box · ECC · BB84 QKD |
| §3 | 🌌 Quantum Info | Qubits · Bloch sphere · Gates · Entanglement · BB84 |
| §4 | 🧙 Wizards 101 | Bessel J_n/Y_n · cylindrical waveguide · spherical harmonics |
| §5 | ⚔️ Min/Max | Game theory minimax · Nash equilibria · gradient descent |
| §6 | 🦆 Procedural | Boids flocking · L-systems · Perlin noise rocks+weeds |
| §7 | 🧠 Fine-tune ML | PyTorch loop · LoRA rank decomp · loss/accuracy curves |
| §8 | 🎯 Torch 3D | 3D tensor batch · timing · quaternion rotations |
| §9 | 🏆 Gamer Science | ELO · hitbox AABB · ballistics · MMR matchmaking |


In [ ]:
import sympy as sp
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
from scipy.special import jv, yv, jn_zeros, spherical_jn
from scipy.linalg import expm
import time, hashlib, math, warnings, random
warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size':10,'figure.dpi':100})
torch.set_default_dtype(torch.float64)
np.random.seed(101)   # Wizards 101
print('✦ Gamer Science — all modules loaded')

---
# §1 ⚡ Applied Physics — EM Circuits

**Maxwell → circuit analog:** displacement current $I_D = C\\,dV/dt$,  
Faraday $V_L = L\\,dI/dt$, Ohm $V_R = IR$.  

**Transmission line:** $Z_0=\\sqrt{L'/C'}$, reflection $\\Gamma=(Z_L-Z_0)/(Z_L+Z_0)$,  
**S-parameters:** $S_{11}=\\Gamma$, $S_{21}=T=1+\\Gamma$ (lossless).

In [ ]:
# ── SymPy: RLC impedance and resonance ────────────────────────────
omega_s, R_s, L_s, C_s = sp.symbols('omega R L C', positive=True)
s_var = sp.Symbol('s')  # Laplace variable

Z_R = R_s
Z_L = sp.I * omega_s * L_s
Z_C = 1 / (sp.I * omega_s * C_s)

# Series RLC
Z_series = Z_R + Z_L + Z_C
Z_series_s = sp.simplify(Z_series)
print('Series RLC impedance:')
from IPython.display import display, Math
display(Math(r'Z_{series} = ' + sp.latex(Z_series_s)))

# Resonance: Im(Z)=0
omega_res = sp.solve(sp.im(Z_series.rewrite(sp.cos)), omega_s)
# analytical: omega_0 = 1/sqrt(LC)
omega_0 = 1/sp.sqrt(L_s*C_s)
print('Resonance frequency:')
display(Math(r'\omega_0 = \frac{1}{\sqrt{LC}} = ' + sp.latex(omega_0)))

# Quality factor
Q_factor = omega_0*L_s / R_s
display(Math(r'Q = \frac{\omega_0 L}{R} = ' + sp.latex(sp.simplify(Q_factor))))

# Transfer function H(s) = Vout/Vin for bandpass (across R)
H_bp = R_s / (R_s + s_var*L_s + 1/(s_var*C_s))
H_bp = sp.simplify(H_bp * s_var*C_s*R_s / (s_var*C_s*R_s))
print('\nBandpass transfer function:')
display(Math(r'H_{BP}(s) = \frac{sRC}{s^2LC + sRC + 1}'))

# ── Numerical: Bode plot ───────────────────────────────────────────
R_v, L_v, C_v = 50.0, 1e-6, 1e-9   # 50Ω, 1μH, 1nF
omega_0_v = 1/np.sqrt(L_v*C_v)
Q_v       = omega_0_v*L_v/R_v
freq_arr  = np.logspace(6, 10, 2000)
omega_arr = 2*np.pi*freq_arr

Z_arr    = R_v + 1j*omega_arr*L_v + 1/(1j*omega_arr*C_v)
H_bp_arr = R_v / Z_arr   # bandpass
H_lp_arr = (1/(1j*omega_arr*C_v)) / Z_arr  # lowpass
H_hp_arr = (1j*omega_arr*L_v) / Z_arr      # highpass

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for H, name, color in [(H_bp_arr,'Bandpass','b'),
                        (H_lp_arr,'Lowpass','r'),
                        (H_hp_arr,'Highpass','g')]:   # loop: Bode
    axes[0].semilogx(freq_arr, 20*np.log10(np.abs(H)+1e-30), lw=2, label=name, color=color)
axes[0].axvline(omega_0_v/(2*np.pi), ls='--', color='k', lw=1, label=f'f₀={omega_0_v/(2*np.pi)/1e6:.1f}MHz')
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('|H| (dB)')
axes[0].set_title(f'RLC Bode  Q={Q_v:.1f}'); axes[0].legend(fontsize=8); axes[0].grid(True,alpha=0.3)

# Transmission line: S11 vs frequency for load ZL
Z0_tl = 50.0
ZL_vals = {'Open (∞Ω)': 1e9, 'Short (0Ω)': 0.01,
            'Matched (50Ω)': 50.0, 'Reactive (50+50j)': 50+50j}
f_tl = np.logspace(6, 10, 1000)
for name, ZL in ZL_vals.items():      # loop: S11 per load
    Gamma = (ZL - Z0_tl) / (ZL + Z0_tl)
    S11   = np.ones(len(f_tl)) * abs(Gamma)
    axes[1].semilogx(f_tl, 20*np.log10(S11+1e-30), lw=2, label=name)
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('|S₁₁| (dB)')
axes[1].set_title('S₁₁ vs load'); axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

# Smith chart
theta_smith = np.linspace(0, 2*np.pi, 400)
axes[2].plot(np.cos(theta_smith), np.sin(theta_smith), 'k-', lw=1.5)
# Resistance circles: r = 0, 0.5, 1, 2
for r_v in [0, 0.5, 1.0, 2.0, 5.0]:  # loop: resistance circles
    cx = r_v/(1+r_v); rad = 1/(1+r_v)
    axes[2].plot(cx + rad*np.cos(theta_smith), rad*np.sin(theta_smith),
                 'b-', lw=0.7, alpha=0.6)
    if r_v < 3: axes[2].text(cx+rad+0.03, 0, f'{r_v}', fontsize=7, color='blue')
# Reactance arcs: x = ±0.5, ±1, ±2
phi_arc = np.linspace(0.01, np.pi-0.01, 200)
for x_v in [0.5, 1.0, 2.0]:           # loop: reactance arcs
    for sign in [1, -1]:
        cx2 = 1; cy2 = sign/x_v; rad2 = 1/abs(x_v)
        xc   = cx2 + rad2*np.cos(phi_arc)
        yc   = cy2 + sign*rad2*np.sin(phi_arc)
        mask = xc**2 + yc**2 <= 1.001
        axes[2].plot(xc[mask], yc[mask], 'r-', lw=0.7, alpha=0.5)
# Mark loads
for name, ZL in ZL_vals.items():
    G = (ZL - Z0_tl)/(ZL + Z0_tl)
    axes[2].plot(np.real(G), np.imag(G), 'o', ms=8, label=name)
axes[2].set_xlim(-1.1,1.1); axes[2].set_ylim(-1.1,1.1)
axes[2].set_aspect('equal'); axes[2].legend(fontsize=7)
axes[2].set_title('Smith Chart'); axes[2].grid(True,alpha=0.2)
plt.suptitle('⚡ Applied EM Circuits',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §2 🔐 Cryptography — RSA · AES S-box · ECC · BB84

**RSA:** choose primes $p,q$; $n=pq$; $\\phi=(p-1)(q-1)$; pick $e$; $d=e^{-1}\\mod\\phi$.  
Encrypt $c=m^e\\mod n$, decrypt $m=c^d\\mod n$.  

**AES S-box:** 8-bit → 8-bit nonlinear substitution via GF(2⁸) inversion + affine.  
**ECC:** points on $y^2=x^3+ax+b\\mod p$; ECDH shared secret via point multiplication.  
**BB84:** quantum key distribution — polarization bases X/Z, intercept-resend attack detected by error rate.

In [ ]:
# ── RSA (small toy example, SymPy exact) ─────────────────────────
def is_prime_simple(n):
    if n < 2: return False
    for i in range(2, int(n**0.5)+1):
        if n%i==0: return False
    return True

def extended_gcd(a, b):
    if a==0: return b,0,1
    g,x,y = extended_gcd(b%a, a)
    return g, y-(b//a)*x, x

def mod_inverse(e, phi):
    g,x,_ = extended_gcd(e%phi, phi)
    if g!=1: raise ValueError('No inverse')
    return x%phi

# RSA-like with small primes for demonstration
p_rsa, q_rsa = 61, 53
n_rsa  = p_rsa * q_rsa
phi_rsa= (p_rsa-1)*(q_rsa-1)
e_rsa  = 17
d_rsa  = mod_inverse(e_rsa, phi_rsa)

print('RSA Key Generation:')
print(f'  p={p_rsa}, q={q_rsa}, n={n_rsa}, φ={phi_rsa}')
print(f'  e={e_rsa} (public), d={d_rsa} (private)')
print(f'  Verify: e·d mod φ = {(e_rsa*d_rsa)%phi_rsa} (must be 1)')

# Encrypt/decrypt loop for multiple messages
messages = [65, 42, 99, 111, 7]
print('\nEncrypt → Decrypt:')
print(f'{"m":5}  {"c=m^e mod n":15}  {"m_dec=c^d mod n":15}  OK?')
for m_v in messages:               # loop: RSA encrypt/decrypt
    c_v   = pow(m_v, e_rsa, n_rsa)
    m_dec = pow(c_v, d_rsa, n_rsa)
    print(f'{m_v:5}  {c_v:15}  {m_dec:15}  {"✓" if m_dec==m_v else "✗"}')

# ── AES S-box (full 256-entry) ─────────────────────────────────────
# GF(2^8) with irreducible poly x^8+x^4+x^3+x+1 (0x11B)
def gf_mul(a, b, mod=0x11B):
    result = 0
    while b:
        if b&1: result ^= a
        a <<= 1
        if a&0x100: a ^= mod
        b >>= 1
    return result & 0xFF

def gf_inv(a, mod=0x11B):
    """Extended Euclidean in GF(2^8)."""
    if a==0: return 0
    r0,r1 = mod,a; s0,s1 = 0,1
    while r1:
        # polynomial long division in GF(2)
        q=0; r0t,r1t=r0,r1
        d=r0t.bit_length()-r1t.bit_length()
        while d>=0:
            if r0t&(1<<(r1t.bit_length()-1+d)):
                r0t^=r1t<<d; q^=1<<d
            d-=1
        r0,r1=r1,r0t; s0,s1=s1,s0^gf_mul(q,s1,mod) if False else (s1, (s0^(q&0xFF)))
    return s0&0xFF

def aes_sbox_entry(b):
    """AES SubBytes: GF(2^8) inversion + affine transform."""
    inv = gf_inv(b)
    # Affine transform matrix (circulant)
    result = 0
    for i in range(8):             # loop: bit-level affine
        bit = ((inv>>i)&1)^((inv>>((i+4)%8))&1)^((inv>>((i+5)%8))&1)^\
              ((inv>>((i+6)%8))&1)^((inv>>((i+7)%8))&1)^((0x63>>i)&1)
        result |= (bit<<i)
    return result

# Compute full AES S-box
AES_SBOX = [aes_sbox_entry(i) for i in range(256)]   # loop: 256 entries
# Verify against known values
known = {0x00:0x63, 0x01:0x7c, 0x02:0x77, 0x53:0xed, 0xff:0x16}
ok = all(AES_SBOX[k]==v for k,v in known.items())
print(f'\nAES S-box computed (256 entries), verification: {"✓" if ok else "✗"}')
print(f'  S[0x00]={AES_SBOX[0x00]:02X}, S[0x01]={AES_SBOX[0x01]:02X}, S[0x53]={AES_SBOX[0x53]:02X}')

# ── ECC over small prime field (secp-like toy) ─────────────────────
class ECCPoint:
    """Elliptic curve point over Fp: y²=x³+ax+b mod p."""
    def __init__(self, x, y, a, b, p):
        self.x,self.y,self.a,self.b,self.p = x,y,a,b,p

    @property
    def is_infinity(self): return self.x is None

    def __add__(self, other):
        if self.is_infinity:  return other
        if other.is_infinity: return self
        p=self.p; a=self.a
        if self.x==other.x and (self.y+other.y)%p==0:
            return ECCPoint(None,None,a,self.b,p)  # point at infinity
        if self.x==other.x:
            lam=(3*self.x**2+a)*pow(2*self.y,p-2,p)%p
        else:
            lam=(other.y-self.y)*pow(other.x-self.x,p-2,p)%p
        xr=(lam**2-self.x-other.x)%p
        yr=(lam*(self.x-xr)-self.y)%p
        return ECCPoint(xr,yr,a,self.b,p)

    def __mul__(self, k):
        result=ECCPoint(None,None,self.a,self.b,self.p); addend=self
        while k>0:                   # loop: double-and-add
            if k&1: result=result+addend
            addend=addend+addend; k>>=1
        return result

    def __repr__(self): return f'({self.x},{self.y})'

# Toy curve: y²=x³+2x+3 mod 97
p_ec=97; a_ec=2; b_ec=3
# Find a point on the curve
G_ec=None
for x_t in range(p_ec):             # loop: find generator
    rhs=(x_t**3+a_ec*x_t+b_ec)%p_ec
    for y_t in range(p_ec):
        if y_t**2%p_ec==rhs: G_ec=ECCPoint(x_t,y_t,a_ec,b_ec,p_ec); break
    if G_ec: break

# ECDH key exchange
k_alice=7; k_bob=11
pub_alice = G_ec*k_alice
pub_bob   = G_ec*k_bob
shared_a  = pub_bob*k_alice
shared_b  = pub_alice*k_bob
print(f'\nECC (y²=x³+{a_ec}x+{b_ec} mod {p_ec}):')
print(f'  Generator G={G_ec}')
print(f'  Alice pub={pub_alice}, Bob pub={pub_bob}')
print(f'  Shared secret: Alice={shared_a}, Bob={shared_b}, match={shared_a.x==shared_b.x} ✓')

# ── Visualizations ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# RSA: encryption as modular exponentiation mapping
m_arr = np.arange(0, 100)
c_arr = np.array([pow(int(m), e_rsa, n_rsa) for m in m_arr])
axes[0].scatter(m_arr, c_arr, c=m_arr, cmap='hsv', s=15)
axes[0].set_xlabel('Plaintext m'); axes[0].set_ylabel('Ciphertext c')
axes[0].set_title(f'RSA encryption map (n={n_rsa},e={e_rsa})')
axes[0].grid(True,alpha=0.3)

# AES S-box heatmap
sbox_arr = np.array(AES_SBOX).reshape(16,16)
im = axes[1].imshow(sbox_arr, cmap='viridis', aspect='auto')
axes[1].set_title('AES S-box (16×16)'); plt.colorbar(im,ax=axes[1])
axes[1].set_xlabel('col nibble'); axes[1].set_ylabel('row nibble')

# ECC curve + points
x_ec = np.arange(p_ec)
curve_pts = []
for x_v in x_ec:                    # loop: find all curve points
    rhs=(x_v**3+a_ec*x_v+b_ec)%p_ec
    for y_v in range(p_ec):
        if y_v**2%p_ec==rhs: curve_pts.append((x_v,y_v))
if curve_pts:
    xs_c,ys_c=zip(*curve_pts)
    axes[2].scatter(xs_c,ys_c,s=4,color='gray',alpha=0.5,label='Curve')
axes[2].scatter([G_ec.x],[G_ec.y],c='green',s=80,zorder=5,label='G')
axes[2].scatter([pub_alice.x],[pub_alice.y],c='blue',s=80,zorder=5,label='Alice')
axes[2].scatter([pub_bob.x],[pub_bob.y],c='red',s=80,zorder=5,label='Bob')
axes[2].scatter([shared_a.x],[shared_a.y],c='gold',s=120,zorder=6,marker='*',label='Shared')
axes[2].set_title(f'ECC y²=x³+{a_ec}x+{b_ec} mod {p_ec}')
axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3)
plt.suptitle('🔐 Cryptography: RSA · AES · ECC',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §3 🌌 Quantum Information — Qubits · Gates · Entanglement · BB84

**Qubit:** $|\\psi\\rangle = \\alpha|0\\rangle+\\beta|1\\rangle$, $|\\alpha|^2+|\\beta|^2=1$.  
**Bloch sphere:** $(\\theta,\\phi)$ → $|\\psi\\rangle=\\cos(\\theta/2)|0\\rangle+e^{i\\phi}\\sin(\\theta/2)|1\\rangle$.  
**Gates:** $H=\\frac{1}{\\sqrt{2}}\\begin{pmatrix}1&1\\\\1&-1\\end{pmatrix}$, Pauli X/Y/Z, CNOT.  
**BB84:** Alice sends random bits in random bases; Bob measures; error rate > 25% → eavesdropper.

In [ ]:
# ── Qubit states and Bloch sphere ─────────────────────────────────
# Pauli matrices
I2 = np.eye(2, dtype=complex)
sx = np.array([[0,1],[1,0]], dtype=complex)     # X (NOT)
sy = np.array([[0,-1j],[1j,0]], dtype=complex)  # Y
sz = np.array([[1,0],[0,-1]], dtype=complex)    # Z (phase)
H_gate = np.array([[1,1],[1,-1]], dtype=complex)/np.sqrt(2)  # Hadamard
T_gate = np.array([[1,0],[0,np.exp(1j*np.pi/4)]], dtype=complex)  # T gate
CNOT   = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

print('Quantum Gates:')
for name, G in [('X (NOT)',sx),('Y',sy),('Z',sz),('H (Hadamard)',H_gate),('T',T_gate)]:
    print(f'  {name}: det={np.linalg.det(G):.3f}, unitary={np.allclose(G@G.conj().T,I2)}')

# Bloch sphere parametric
def bloch(theta, phi):
    return np.array([np.sin(theta)*np.cos(phi),
                     np.sin(theta)*np.sin(phi),
                     np.cos(theta)])

def qubit(theta, phi):
    return np.array([np.cos(theta/2),
                     np.exp(1j*phi)*np.sin(theta/2)], dtype=complex)

# Famous states
states = {
    '|0⟩':  (0,     0),
    '|1⟩':  (np.pi, 0),
    '|+⟩':  (np.pi/2, 0),
    '|-⟩':  (np.pi/2, np.pi),
    '|i⟩':  (np.pi/2, np.pi/2),
    '|-i⟩': (np.pi/2, 3*np.pi/2),
    'T|+⟩': None,  # computed below
}

# Compute T|+⟩
psi_plus = qubit(np.pi/2, 0)
psi_T    = T_gate @ psi_plus
# psi_T to Bloch coords
a_T,b_T = psi_T
theta_T  = 2*np.arccos(np.abs(a_T).clip(0,1))
phi_T    = np.angle(b_T) - np.angle(a_T)
states['T|+⟩'] = (theta_T, phi_T)

fig = plt.figure(figsize=(15,5))
ax_bloch = fig.add_subplot(131, projection='3d')
# Draw sphere
u_s,v_s = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
xs_b=np.sin(v_s)*np.cos(u_s); ys_b=np.sin(v_s)*np.sin(u_s); zs_b=np.cos(v_s)
ax_bloch.plot_surface(xs_b,ys_b,zs_b,alpha=0.08,color='cyan')
ax_bloch.plot(np.cos(u_s[0]),np.sin(u_s[0]),np.zeros_like(u_s[0]),'k-',lw=0.5,alpha=0.3)
for name_s,(th,ph) in states.items():   # loop: plot states
    bv = bloch(th,ph)
    ax_bloch.quiver(0,0,0,*bv,length=0.95,arrow_length_ratio=0.15,lw=2)
    ax_bloch.text(*(bv*1.15), name_s, fontsize=8)
ax_bloch.set_box_aspect([1,1,1])
ax_bloch.set_title('Bloch Sphere'); ax_bloch.set_xlabel('X'); ax_bloch.set_ylabel('Y')

# Bell states and entanglement
ax_ent = fig.add_subplot(132)
# Bell basis: Phi+, Phi-, Psi+, Psi-
PhiP = np.array([1,0,0,1],dtype=complex)/np.sqrt(2)
PhiM = np.array([1,0,0,-1],dtype=complex)/np.sqrt(2)
PsiP = np.array([0,1,1,0],dtype=complex)/np.sqrt(2)
PsiM = np.array([0,1,-1,0],dtype=complex)/np.sqrt(2)

# Density matrix of Phi+
rho_PhiP = np.outer(PhiP, PhiP.conj())
# Partial trace (trace out qubit 2) → rho_A
rho_A = rho_PhiP[:2,:2] + rho_PhiP[2:,2:]
eigs = np.real(np.linalg.eigvalsh(rho_A))
entropy_A = -sum(e*np.log2(e+1e-30) for e in eigs if e>1e-10)

im_dm = ax_ent.imshow(np.abs(rho_PhiP), cmap='hot', vmin=0, vmax=0.5)
ax_ent.set_title(f'|Φ⁺⟩ density matrix\nEntanglement entropy S={entropy_A:.3f} bits')
ax_ent.set_xlabel('Basis state'); ax_ent.set_ylabel('Basis state')
for i in range(4):                  # loop: label matrix entries
    for j in range(4):
        ax_ent.text(j,i,f'{np.abs(rho_PhiP[i,j]):.2f}',ha='center',va='center',fontsize=8)
plt.colorbar(im_dm, ax=ax_ent)

# ── BB84 simulation ─────────────────────────────────────────────────
ax_bb = fig.add_subplot(133)
N_bb84 = 1000
rng    = np.random.default_rng(84)

def bb84_sim(n_bits, eavesdrop_fraction=0.0):
    # Alice: random bits + random bases (0=Z, 1=X)
    alice_bits  = rng.integers(0,2,n_bits)
    alice_bases = rng.integers(0,2,n_bits)
    # Eavesdropper: intercepts fraction, measures in random basis
    eve_intercept = rng.random(n_bits) < eavesdrop_fraction
    eve_bases     = rng.integers(0,2,n_bits)
    # Bob: random bases
    bob_bases = rng.integers(0,2,n_bits)
    # Sifting: keep where Alice and Bob chose same basis
    sift_mask = alice_bases == bob_bases
    # Errors introduced by Eve when she guessed wrong basis
    error_mask = sift_mask & eve_intercept & (eve_bases != alice_bases)
    n_sifted   = sift_mask.sum()
    n_errors   = error_mask.sum()
    qber       = n_errors/max(n_sifted,1)
    return n_sifted, n_errors, qber

eve_fracs = np.linspace(0, 1, 50)
qbers     = [bb84_sim(N_bb84, f)[2] for f in eve_fracs]   # loop: QBER vs Eve
ax_bb.plot(eve_fracs*100, np.array(qbers)*100, 'r-o', ms=4, lw=2)
ax_bb.axhline(11, ls='--', color='k', lw=1.5, label='Security threshold 11%')
ax_bb.axhline(25, ls=':', color='gray', lw=1, label='Max Eve QBER 25%')
ax_bb.set_xlabel('Eve intercept fraction (%)')
ax_bb.set_ylabel('QBER (%)')
ax_bb.set_title(f'BB84 QBER vs eavesdropping\n(N={N_bb84} bits)')
ax_bb.legend(fontsize=8); ax_bb.grid(True,alpha=0.3)

plt.suptitle('🌌 Quantum Information: Bloch · Bell · BB84',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §4 🧙 Wizards 101 — Bessel Functions · Cylindrical Modes · Spherical Harmonics

**Bessel equation:** $x^2y''+xy'+(x^2-n^2)y=0$  
Solutions: $J_n(x)$ (finite at 0), $Y_n(x)$ (singular at 0).  

**Cylindrical waveguide TE modes:** $E_z=0$, $H_z\\propto J_m(k_c r)\\cos(m\\phi)e^{i\\beta z}$  
where $k_c=j_{mn}/a$ from zeros of $J_m$.  

**Spherical harmonics** $Y_\\ell^m(\\theta,\\phi)$ — angular part of hydrogen atom wavefunctions.

In [ ]:
# ── SymPy: Bessel ODE and recurrence relations ────────────────────
x_b, n_b = sp.Symbol('x', positive=True), sp.Symbol('n', nonneg=True)
J_n = sp.besselj(n_b, x_b)
Y_n = sp.bessely(n_b, x_b)

print('Bessel recurrence relations:')
# J_{n-1}(x) + J_{n+1}(x) = (2n/x)*J_n(x)
display(Math(r'J_{n-1}(x)+J_{n+1}(x) = \frac{2n}{x}J_n(x)'))
display(Math(r"J_n'(x) = \frac{1}{2}(J_{n-1}(x)-J_{n+1}(x))"))
display(Math(r'\int_0^a J_n(k_{mn}r/a)J_n(k_{mn\prime}r/a)r\,dr = '
             r'\frac{a^2}{2}J_{n+1}^2(k_{mn})\delta_{mm\prime}  \text{ (orthogonality)}'))

# ── Numerical: J_n, Y_n for n=0..5 ───────────────────────────────
x_arr = np.linspace(0.01, 20, 2000)
n_orders = range(6)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# J_n and Y_n vs x
ax_j = axes[0][0]; ax_y = axes[0][1]
colors_b = plt.cm.tab10(np.linspace(0,0.6,6))
for n_v, col in zip(n_orders, colors_b):   # loop: plot J_n, Y_n
    ax_j.plot(x_arr, jv(n_v, x_arr), color=col, lw=1.5, label=f'J_{n_v}')
    ax_y.plot(x_arr, yv(n_v, x_arr).clip(-2,2), color=col, lw=1.5, label=f'Y_{n_v}')
ax_j.axhline(0, color='k', lw=0.5); ax_j.set_ylim(-0.5, 1.1)
ax_j.set_title('Bessel J_n(x)'); ax_j.legend(fontsize=8); ax_j.grid(True,alpha=0.3)
ax_y.axhline(0, color='k', lw=0.5); ax_y.set_ylim(-2, 0.6)
ax_y.set_title('Bessel Y_n(x)'); ax_y.legend(fontsize=8); ax_y.grid(True,alpha=0.3)

# Cylindrical waveguide TE modes: field pattern J_m(k_mn r)
ax_wg = axes[0][2]
r_arr  = np.linspace(0, 1, 300)
# First few zeros of J_0, J_1, J_2
mode_info = []
for m_v in range(3):                # loop: TE modes
    zeros = jn_zeros(m_v, 4)
    for i_z, z_v in enumerate(zeros[:2]):
        label = f'TE_{m_v}{i_z+1}: j_{m_v}{i_z+1}={z_v:.3f}'
        field  = jv(m_v, z_v * r_arr)
        ax_wg.plot(r_arr, field, lw=1.5, label=label)
        mode_info.append((m_v, i_z+1, z_v))
ax_wg.axhline(0, color='k', lw=0.5)
ax_wg.set_xlabel('r/a'); ax_wg.set_ylabel('J_m(j_mn r/a)')
ax_wg.set_title('Cylindrical waveguide modes'); ax_wg.legend(fontsize=7); ax_wg.grid(True,alpha=0.3)

# 2D mode patterns
r2d = np.linspace(0, 1, 150); phi2d = np.linspace(0, 2*np.pi, 150)
R2D, PHI2D = np.meshgrid(r2d, phi2d)
X2D = R2D*np.cos(PHI2D); Y2D = R2D*np.sin(PHI2D)

mode_plots = [(0,1,jn_zeros(0,1)[0]), (1,1,jn_zeros(1,1)[0]), (2,1,jn_zeros(2,1)[0])]
for idx,(m_v,n_v,z_v) in enumerate(mode_plots):  # loop: 2D mode maps
    ax2d = axes[1][idx]
    field_2d = jv(m_v, z_v*R2D) * np.cos(m_v*PHI2D)
    im2 = ax2d.contourf(X2D, Y2D, field_2d, levels=20, cmap='RdBu')
    # draw waveguide wall
    th_circ = np.linspace(0,2*np.pi,200)
    ax2d.plot(np.cos(th_circ),np.sin(th_circ),'k-',lw=2)
    ax2d.set_aspect('equal')
    ax2d.set_title(f'TE_{m_v}{n_v} mode  j={z_v:.3f}')
    ax2d.axis('off'); plt.colorbar(im2,ax=ax2d,shrink=0.8)

plt.suptitle('🧙 Wizards 101: Bessel Functions & Cylindrical Waveguides',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

# ── Print zeros table ──────────────────────────────────────────────
print('\nBessel function zeros j_{mn} (cutoff wavenumbers k_c = j_mn/a):')
print(f'{"":8}', end='')
for n_v in range(1,5): print(f'  n={n_v}    ',end='')
print()
for m_v in range(5):               # loop: print zero table
    zeros = jn_zeros(m_v,4)
    print(f'm={m_v}:    ',end='')
    for z in zeros: print(f'{z:9.4f} ',end='')
    print()

---
# §5 ⚔️ Numerical Min/Max — Game Theory · Minimax · Nash · Gradient Descent

**Minimax theorem** (von Neumann): in a zero-sum game, $\\max_x\\min_y f(x,y)=\\min_y\\max_x f(x,y)$.  
**Nash equilibrium:** no player can improve by unilateral deviation.  
**Gradient descent** (Roblox min), **gradient ascent** (Fortnite max).

In [ ]:
# ── SymPy: minimax theorem setup ─────────────────────────────────
x_mm, y_mm = sp.symbols('x y', real=True)
# Saddle function: f = x^2 - y^2
f_saddle = x_mm**2 - y_mm**2
critical = sp.solve([sp.diff(f_saddle,x_mm), sp.diff(f_saddle,y_mm)])
print('Saddle point of f = x² - y²:')
display(Math(r'f(x,y)=x^2-y^2,\quad \nabla f=0 \Rightarrow '+sp.latex(critical)))

# Hessian
H_mm = sp.Matrix([[sp.diff(f_saddle,x_mm,2), sp.diff(f_saddle,x_mm,y_mm)],
                  [sp.diff(f_saddle,y_mm,x_mm), sp.diff(f_saddle,y_mm,2)]])
print('Hessian (indefinite → saddle):'); display(Math(r'H = '+sp.latex(H_mm)))
print('Eigenvalues:', sp.eigenvals(H_mm))

# ── 2×2 game theory: rock-paper-scissors ──────────────────────────
# Payoff matrix for player A
A_rps = np.array([[ 0,-1, 1],
                  [ 1, 0,-1],
                  [-1, 1, 0]], dtype=float)
# Nash equilibrium: uniform mixed strategy p=q=(1/3,1/3,1/3)
p_ne = np.array([1/3,1/3,1/3])
q_ne = p_ne
print(f'\nRock-Paper-Scissors Nash value: {p_ne @ A_rps @ q_ne:.6f} (should be 0)')

# General 2-player zero-sum via linear programming (simplex)
def solve_game_lp(A_mat):
    """Find Nash equilibrium via LP (value iteration). Returns (p*, q*, v)."""
    m,n = A_mat.shape
    # Row player maximizes min expected payoff
    # Use fictitious play iteration
    p = np.ones(m)/m; q = np.ones(n)/n
    for t in range(5000):           # loop: fictitious play
        # Best response for row player
        row_payoffs = A_mat @ q
        p_br = np.zeros(m); p_br[np.argmax(row_payoffs)] = 1.0
        # Best response for col player
        col_payoffs = p @ A_mat
        q_br = np.zeros(n); q_br[np.argmin(col_payoffs)] = 1.0
        lr = 1.0/(t+2)
        p = (1-lr)*p + lr*p_br
        q = (1-lr)*q + lr*q_br
    v = p @ A_mat @ q
    return p, q, v

# Prisoner's dilemma payoff matrix
A_pd = np.array([[-1,-3],[ 0,-2]], dtype=float)
p_pd, q_pd, v_pd = solve_game_lp(A_pd)
print(f"Prisoner's dilemma Nash: p={p_pd.round(3)}, q={q_pd.round(3)}, v={v_pd:.3f}")

# ── Gradient descent (Roblox min) vs ascent (Fortnite max) ────────
def rosenbrock(x,y): return (1-x)**2 + 100*(y-x**2)**2
def rosenbrock_grad(x,y):
    gx = -2*(1-x) - 400*x*(y-x**2)
    gy = 200*(y-x**2)
    return np.array([gx,gy])

# GD with momentum (Roblox build: minimize the loss)
pos_gd = np.array([-1.0, 1.0])
vel_gd = np.zeros(2); lr_gd=1e-3; mom=0.9
path_gd = [pos_gd.copy()]
for _ in range(3000):               # loop: gradient descent
    g = rosenbrock_grad(*pos_gd)
    vel_gd = mom*vel_gd - lr_gd*g
    pos_gd = pos_gd + vel_gd
    path_gd.append(pos_gd.copy())
path_gd = np.array(path_gd)

# Adam optimizer (Fortnite max: maximize negative = minimize)
pos_adam = np.array([-1.0, 1.0])
m1,m2=np.zeros(2),np.zeros(2); b1,b2=0.9,0.999; eps_a=1e-8; lr_a=1e-2
path_adam = [pos_adam.copy()]
for t_a in range(1,3001):           # loop: Adam
    g_a = rosenbrock_grad(*pos_adam)
    m1  = b1*m1 + (1-b1)*g_a
    m2  = b2*m2 + (1-b2)*g_a**2
    m1h = m1/(1-b1**t_a); m2h = m2/(1-b2**t_a)
    pos_adam = pos_adam - lr_a*m1h/(np.sqrt(m2h)+eps_a)
    path_adam.append(pos_adam.copy())
path_adam = np.array(path_adam)

# ── Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15,4))

# Saddle point
xg,yg=np.meshgrid(np.linspace(-2,2,100),np.linspace(-2,2,100))
zg=xg**2-yg**2
axes[0].contourf(xg,yg,zg,levels=20,cmap='RdBu')
axes[0].contour(xg,yg,zg,levels=20,colors='k',linewidths=0.3)
axes[0].plot(0,0,'w*',ms=15,label='Saddle (0,0)')
axes[0].set_title('Saddle f=x²−y² (minimax point)')
axes[0].legend(fontsize=9); axes[0].grid(True,alpha=0.2)

# Rosenbrock + optimization paths
xr,yr=np.meshgrid(np.linspace(-2,2,200),np.linspace(-1,3,200))
zr=np.log1p(rosenbrock(xr,yr))
axes[1].contourf(xr,yr,zr,levels=30,cmap='hot')
axes[1].plot(path_gd[:,0],path_gd[:,1],'b-',lw=0.8,alpha=0.7,label='SGD+momentum')
axes[1].plot(path_adam[:,0],path_adam[:,1],'g-',lw=0.8,alpha=0.7,label='Adam')
axes[1].plot(1,1,'w*',ms=15,label='Min (1,1)')
axes[1].set_xlim(-2,2); axes[1].set_ylim(-1,3)
axes[1].set_title('Rosenbrock: SGD vs Adam'); axes[1].legend(fontsize=8)

# Loss curves
losses_gd   = [rosenbrock(*p) for p in path_gd[::10]]    # loop: loss
losses_adam = [rosenbrock(*p) for p in path_adam[::10]]
t_plot = np.arange(len(losses_gd))*10
axes[2].semilogy(t_plot,losses_gd,'b-',lw=2,label='SGD+momentum')
axes[2].semilogy(np.arange(len(losses_adam))*10,losses_adam,'g-',lw=2,label='Adam')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Loss (log)')
axes[2].set_title('Convergence: Roblox MIN → Fortnite MAX')
axes[2].legend(); axes[2].grid(True,alpha=0.3)
plt.suptitle('⚔️ Min/Max: Game Theory · Optimization',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §6 🦆 Procedural — Boids Flocking · L-systems · Perlin Noise

**Boids** (Reynolds 1987): three rules — separation, alignment, cohesion.  
**L-systems** (Lindenmayer): string rewriting → fractal plants/weeds.  
**Perlin noise**: coherent gradient noise → rocky terrain, stone textures.

In [ ]:
# ── Boids flocking (duck & geese) ────────────────────────────────
N_birds = 80; BOX = 100.0
pos_b = np.random.rand(N_birds,2) * BOX
vel_b = (np.random.rand(N_birds,2)-0.5) * 4

# Labels: first 20 = ducks, rest = geese
species = np.array(['🦆']*20 + ['🪿']*(N_birds-20))

def boids_step(pos, vel, dt=0.2, r_sep=3.0, r_align=10.0, r_coh=20.0,
               w_sep=1.8, w_ali=1.0, w_coh=0.5, speed_max=5.0):
    N = len(pos); new_vel = vel.copy()
    diff = pos[:,None,:] - pos[None,:,:]  # (N,N,2) pairwise differences
    dist = np.sqrt((diff**2).sum(-1)) + np.eye(N)*1e9  # (N,N)

    for i in range(N):                    # loop: per-bird rules
        # Separation
        mask_sep = dist[i] < r_sep
        if mask_sep.sum()>0:
            new_vel[i] += w_sep * diff[i][mask_sep].sum(0)  # push away
        # Alignment
        mask_ali = dist[i] < r_align
        if mask_ali.sum()>0:
            new_vel[i] += w_ali * (vel[mask_ali].mean(0) - vel[i])
        # Cohesion
        mask_coh = dist[i] < r_coh
        if mask_coh.sum()>0:
            new_vel[i] += w_coh * (pos[mask_coh].mean(0) - pos[i])

    # Limit speed
    speeds = np.linalg.norm(new_vel, axis=1, keepdims=True).clip(0.1)
    new_vel = new_vel / speeds * np.minimum(speeds, speed_max)
    new_pos = (pos + new_vel*dt) % BOX
    return new_pos, new_vel

# Run simulation
N_steps = 200; snapshots = []
for step in range(N_steps):             # loop: boids simulation
    pos_b, vel_b = boids_step(pos_b, vel_b)
    if step in [0,50,100,150,199]: snapshots.append(pos_b.copy())

# ── L-system weeds ────────────────────────────────────────────────
def lsystem_expand(axiom, rules, n_iters):
    s = axiom
    for _ in range(n_iters):            # loop: L-system expansion
        s = ''.join(rules.get(c,c) for c in s)
    return s

def lsystem_draw(cmds, angle_deg=25, step=3):
    """Turtle graphics: F=forward, +=turn left, -=turn right, [=push, ]=pop"""
    angle = np.radians(angle_deg)
    pos   = np.array([0.0,0.0]); heading=np.pi/2
    lines = []; stack = []
    for c in cmds:
        if c in ('F','G'):
            new_pos = pos + step*np.array([np.cos(heading),np.sin(heading)])
            lines.append((pos.copy(), new_pos.copy()))
            pos = new_pos
        elif c=='+': heading += angle
        elif c=='-': heading -= angle
        elif c=='[': stack.append((pos.copy(), heading))
        elif c==']':
            if stack: pos,heading=stack.pop()
    return lines

# Classic weed: Prusinkiewicz Fig 1.24c
rules_weed = {'F':'FF','X':'F+[[X]-X]-F[-FX]+X'}
weed_str   = lsystem_expand('X', rules_weed, 5)
weed_lines = lsystem_draw(weed_str, angle_deg=25, step=2)

# Koch snowflake (rocks)
rules_koch = {'F':'F+F--F+F'}
koch_str   = lsystem_expand('F--F--F', rules_koch, 4)
koch_lines = lsystem_draw(koch_str, angle_deg=60, step=4)

# ── Perlin noise terrain ────────────────────────────────────────────
def fade(t):  return 6*t**5 - 15*t**4 + 10*t**3
def lerp(a,b,t): return a + t*(b-a)

def perlin2d(x,y,seed=0):
    rng2 = np.random.default_rng(seed)
    # Gradient table
    angles = rng2.uniform(0,2*np.pi, 256)
    grads  = np.stack([np.cos(angles),np.sin(angles)],1)
    perm   = rng2.permutation(256)

    xi=x.astype(int)&255; yi=y.astype(int)&255
    xf=x-x.astype(int);  yf=y-y.astype(int)
    u=fade(xf); v=fade(yf)

    def grad2(ix,iy,dx,dy):
        g=grads[(perm[ix&255]+iy)&255]
        return g[:,0]*dx + g[:,1]*dy

    n00=grad2(xi,   yi,   xf,   yf)
    n10=grad2(xi+1, yi,   xf-1, yf)
    n01=grad2(xi,   yi+1, xf,   yf-1)
    n11=grad2(xi+1, yi+1, xf-1, yf-1)
    return lerp(lerp(n00,n10,u), lerp(n01,n11,u), v)

# Multi-octave Perlin for rocky terrain
res = 256
gx,gy = np.meshgrid(np.linspace(0,8,res),np.linspace(0,8,res))
gx_f=gx.ravel(); gy_f=gy.ravel()
terrain = np.zeros(len(gx_f))
for oct_i,(freq,amp) in enumerate([(1,1.0),(2,0.5),(4,0.25),(8,0.125)]):
    terrain += amp * perlin2d(gx_f*freq, gy_f*freq, seed=oct_i)
terrain = terrain.reshape(res,res)

# ── Plot all ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(15,5))
ax_boid = fig.add_subplot(131)
# Final boids snapshot with velocity arrows
colors_boid = ['#1f77b4']*20 + ['#ff7f0e']*60
ax_boid.scatter(snapshots[-1][:,0], snapshots[-1][:,1],
                c=colors_boid, s=20, alpha=0.8)
ax_boid.quiver(snapshots[-1][:,0], snapshots[-1][:,1],
               vel_b[:,0], vel_b[:,1],
               color=colors_boid, scale=60, alpha=0.5)
ax_boid.set_xlim(0,BOX); ax_boid.set_ylim(0,BOX)
ax_boid.set_title(f'Boids: Ducks (blue) + Geese (orange)\n{N_steps} steps')
ax_boid.set_aspect('equal'); ax_boid.grid(True,alpha=0.2)

ax_lsys = fig.add_subplot(132)
for p1,p2 in weed_lines[:3000]:    # loop: draw weed lines
    ax_lsys.plot([p1[0],p2[0]],[p1[1],p2[1]],'g-',lw=0.4,alpha=0.7)
ax_lsys.set_title(f'L-system weed (5 iters, {len(weed_str)} chars)')
ax_lsys.set_aspect('equal'); ax_lsys.axis('off')

ax_ter = fig.add_subplot(133)
ax_ter.imshow(terrain, cmap='terrain', origin='lower')
ax_ter.contour(terrain, levels=10, colors='k', linewidths=0.3, alpha=0.4)
ax_ter.set_title('Perlin noise rocky terrain (4 octaves)')
ax_ter.axis('off')
plt.suptitle('🦆 Procedural: Boids · L-systems · Perlin',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §7 🧠 Fine-tune ML — PyTorch Loop · LoRA · Loss/Accuracy Curves

**Transfer learning:** freeze backbone, train head on new task.  
**LoRA** (Low-Rank Adaptation): $W \\leftarrow W_0 + BA$, where $B\\in\\mathbb{R}^{d\\times r}$, $A\\in\\mathbb{R}^{r\\times k}$, $r\\ll\\min(d,k)$.  
Parameter savings: $dk$ → $r(d+k)$, ratio $= r(d+k)/(dk)$.

In [ ]:
# ── SymPy: LoRA parameter analysis ───────────────────────────────
d_sym, k_sym, r_sym = sp.symbols('d k r', positive=True)
params_full  = d_sym * k_sym
params_lora  = r_sym * (d_sym + k_sym)
compression  = params_lora / params_full
print('LoRA compression ratio:')
display(Math(r'\frac{r(d+k)}{dk} = ' + sp.latex(compression)))
# For d=k: ratio = 2r/d
ratio_square = compression.subs(k_sym, d_sym)
display(Math(r'd=k: \frac{2r}{d} = '+sp.latex(sp.simplify(ratio_square))))

# Numerical: typical LLM layer (d=4096, k=4096)
d_v,k_v=4096,4096
print('\nLoRA for 4096×4096 weight matrix:')
for r_v in [1,2,4,8,16,32,64]:      # loop: ranks
    p_full = d_v*k_v; p_lora=r_v*(d_v+k_v)
    pct=100*p_lora/p_full
    print(f'  r={r_v:3d}: {p_lora:8,} params  ({pct:.2f}% of full  {p_full//p_lora}× compression)')

# ── PyTorch: tiny model fine-tuning simulation ─────────────────────
class TinyNet(nn.Module):
    """Small CNN for 32×32 image classification."""
    def __init__(self, n_classes=10):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),  # →16×16
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(), nn.MaxPool2d(2),  # →8×8
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(), nn.AdaptiveAvgPool2d(4), # →4×4
        )
        self.head = nn.Linear(64*4*4, n_classes)

    def forward(self,x):
        feat = self.backbone(x).flatten(1)
        return self.head(feat)

class LoRALinear(nn.Module):
    """Wraps a frozen Linear layer with LoRA adapters A, B."""
    def __init__(self, linear, rank=4):
        super().__init__()
        self.linear = linear
        for p in self.linear.parameters(): p.requires_grad_(False)  # freeze
        d_out,d_in = linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, d_in, dtype=torch.float64)*0.01)
        self.B = nn.Parameter(torch.zeros(d_out, rank, dtype=torch.float64))
        self.rank = rank

    def forward(self,x):
        return self.linear(x) + (x @ self.A.T @ self.B.T)

# Build model + apply LoRA
torch.manual_seed(42)
model_pretrained = TinyNet(10)
model_lora       = TinyNet(10)
model_lora.load_state_dict(model_pretrained.state_dict())  # same init
# Freeze backbone, wrap head with LoRA
for p in model_lora.backbone.parameters(): p.requires_grad_(False)
model_lora.head = LoRALinear(model_lora.head, rank=8)

# Count trainable params
def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

t_ft,  tr_ft  = count_params(model_pretrained)
t_lora,tr_lora= count_params(model_lora)
print(f'\nFull fine-tune:  {tr_ft:,} trainable / {t_ft:,} total ({100*tr_ft/t_ft:.1f}%)')
print(f'LoRA (rank=8):   {tr_lora:,} trainable / {t_lora:,} total ({100*tr_lora/t_lora:.1f}%)')
print(f'LoRA is {tr_ft//max(tr_lora,1)}× fewer trainable params')

# Synthetic training loop (random data, track loss/accuracy)
def train_loop(model, n_epochs=60, lr=1e-3, batch=64, n_classes=10):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses=[]; accs=[]
    model.train()
    for ep in range(n_epochs):         # loop: training epochs
        # Synthetic batch
        x_batch = torch.randn(batch,1,32,32,dtype=torch.float64)
        y_batch = torch.randint(0,n_classes,(batch,))
        opt.zero_grad()
        logits = model(x_batch)
        loss   = criterion(logits,y_batch)
        loss.backward()
        opt.step()
        # Validation
        with torch.no_grad():
            x_val=torch.randn(200,1,32,32,dtype=torch.float64)
            y_val=torch.randint(0,n_classes,(200,))
            logits_v=model(x_val)
            loss_v = criterion(logits_v,y_val).item()
            acc_v  = (logits_v.argmax(1)==y_val).float().mean().item()
        losses.append(loss_v); accs.append(acc_v)
    return losses, accs

print('\nTraining full fine-tune...')
losses_full, accs_full = train_loop(model_pretrained)
print('Training LoRA...')
losses_lora, accs_lora = train_loop(model_lora)

fig, axes = plt.subplots(1, 3, figsize=(14,4))
ep_arr = np.arange(60)
axes[0].plot(ep_arr, losses_full,'b-',lw=2,label='Full fine-tune')
axes[0].plot(ep_arr, losses_lora,'r--',lw=2,label='LoRA (rank=8)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val loss')
axes[0].set_title('Loss curves'); axes[0].legend(); axes[0].grid(True,alpha=0.3)

axes[1].plot(ep_arr, np.array(accs_full)*100,'b-',lw=2,label='Full')
axes[1].plot(ep_arr, np.array(accs_lora)*100,'r--',lw=2,label='LoRA')
axes[1].axhline(10,ls=':',color='k',lw=1,label='Random (10%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val accuracy (%)')
axes[1].set_title('Accuracy curves'); axes[1].legend(); axes[1].grid(True,alpha=0.3)

# LoRA rank study
ranks_study=[1,2,4,8,16,32]
final_losses=[]
for r_v in ranks_study:             # loop: rank ablation
    m_tmp=TinyNet(10)
    for p in m_tmp.backbone.parameters(): p.requires_grad_(False)
    m_tmp.head=LoRALinear(m_tmp.head,rank=r_v)
    l_tmp,_=train_loop(m_tmp,n_epochs=30)
    final_losses.append(l_tmp[-1])
axes[2].plot(ranks_study,final_losses,'go-',ms=8,lw=2)
axes[2].set_xlabel('LoRA rank r'); axes[2].set_ylabel('Final val loss')
axes[2].set_title('LoRA rank ablation'); axes[2].grid(True,alpha=0.3)
plt.suptitle('🧠 Fine-tune ML: Full vs LoRA',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §8 🎯 Torch 3D — Quaternions · Batch Transforms · Timing

Quaternion rotation $q = w + xi + yj + zk$, $|q|=1$:  
$$\\mathbf{v}' = q \\mathbf{v} q^* = R(q)\\mathbf{v}$$  
Matrix form: $R_{ij} = (w^2+x^2-y^2-z^2)\\delta_{ij} + 2(q_iq_j - \\epsilon_{ijk}wq_k)$

In [ ]:
# ── Torch quaternion utilities ────────────────────────────────────
def quat_mul(q1, q2):
    """Quaternion product. q=(w,x,y,z), shape (...,4)."""
    w1,x1,y1,z1 = q1[...,0],q1[...,1],q1[...,2],q1[...,3]
    w2,x2,y2,z2 = q2[...,0],q2[...,1],q2[...,2],q2[...,3]
    return torch.stack([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ], dim=-1)

def quat_rotate(q, v):
    """Rotate vector v by quaternion q. v shape (...,3)."""
    # Extend v to pure quaternion (0, vx, vy, vz)
    v_q = torch.cat([torch.zeros(*v.shape[:-1],1,dtype=v.dtype), v], dim=-1)
    q_conj = q * torch.tensor([1.,-1.,-1.,-1.],dtype=q.dtype)
    return quat_mul(quat_mul(q,v_q), q_conj)[...,1:]

def axis_angle_to_quat(axis, angle):
    """axis: (...,3) unit vector, angle: (...,) radians → (...,4) quaternion."""
    axis = axis / (axis.norm(dim=-1,keepdim=True)+1e-10)
    ha   = angle.unsqueeze(-1)/2
    return torch.cat([torch.cos(ha), axis*torch.sin(ha)], dim=-1)

def quat_to_matrix(q):
    """q (...,4) → R (...,3,3) rotation matrix."""
    w,x,y,z = q[...,0],q[...,1],q[...,2],q[...,3]
    R = torch.stack([
        torch.stack([1-2*(y**2+z**2), 2*(x*y-z*w), 2*(x*z+y*w)],dim=-1),
        torch.stack([2*(x*y+z*w), 1-2*(x**2+z**2), 2*(y*z-x*w)],dim=-1),
        torch.stack([2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x**2+y**2)],dim=-1),
    ], dim=-2)
    return R

# ── Batch 3D benchmark ────────────────────────────────────────────
batch_sizes = [100, 1000, 10000, 100000, 500000]
timing_results = {}

ops = [
    ('quat_rotate',   lambda N: quat_rotate(
        torch.nn.functional.normalize(torch.randn(N,4,dtype=torch.float64),dim=-1),
        torch.randn(N,3,dtype=torch.float64))),
    ('quat_mul',      lambda N: quat_mul(
        torch.nn.functional.normalize(torch.randn(N,4,dtype=torch.float64),dim=-1),
        torch.nn.functional.normalize(torch.randn(N,4,dtype=torch.float64),dim=-1))),
    ('quat_to_mat',   lambda N: quat_to_matrix(
        torch.nn.functional.normalize(torch.randn(N,4,dtype=torch.float64),dim=-1))),
    ('bmm_3x3',       lambda N: torch.bmm(
        torch.randn(N,3,3,dtype=torch.float64), torch.randn(N,3,3,dtype=torch.float64))),
    ('cross_product',  lambda N: torch.linalg.cross(
        torch.randn(N,3,dtype=torch.float64), torch.randn(N,3,dtype=torch.float64))),
]

print('Torch 3D Batch Timing (ms):')
print(f'{"Op":16}  ' + '  '.join(f'{B:>8}' for B in batch_sizes))
print('-'*70)
for op_name, op_fn in ops:          # loop: benchmark ops
    times_row = []
    for N_v in batch_sizes:         # loop: batch sizes
        torch.manual_seed(0)
        # Warmup
        _ = op_fn(min(N_v,1000))
        t0 = time.perf_counter()
        for _ in range(3): result = op_fn(N_v)
        t_ms = (time.perf_counter()-t0)/3*1000
        times_row.append(t_ms)
    timing_results[op_name] = times_row
    print(f'{op_name:16}  ' + '  '.join(f'{t:8.2f}' for t in times_row))

# Verify quaternion rotation correctness
torch.manual_seed(7)
axis_v  = torch.tensor([[0.,0.,1.]])
angle_v = torch.tensor([np.pi/2])
q_test  = axis_angle_to_quat(axis_v, angle_v)
v_test  = torch.tensor([[1.,0.,0.]])
v_rot   = quat_rotate(q_test, v_test)
print(f'\nQuaternion sanity: rotate (1,0,0) by 90° around Z')
print(f'  Expected: (0,1,0)  Got: ({v_rot[0,0]:.4f},{v_rot[0,1]:.4f},{v_rot[0,2]:.4f})')

# Slerp (spherical linear interpolation for smooth 3D animation)
def slerp(q1, q2, t):
    """SLERP between q1 and q2 at t∈[0,1]."""
    dot = (q1*q2).sum(-1).clamp(-1,1)
    theta = torch.acos(dot.abs())
    sin_theta = torch.sin(theta).clamp(1e-10)
    s1 = torch.sin((1-t)*theta)/sin_theta
    s2 = torch.sin(t*theta)/sin_theta
    sign = torch.sign(dot)
    return s1.unsqueeze(-1)*q1 + (s2*sign).unsqueeze(-1)*q2

# Visualize: 3D rotation path via slerp
q_start = axis_angle_to_quat(torch.tensor([[0.,0.,1.]]), torch.tensor([0.]))
q_end   = axis_angle_to_quat(torch.tensor([[1.,1.,0.]])/np.sqrt(2), torch.tensor([np.pi]))
t_interp= torch.linspace(0,1,100)
q_slerp = torch.stack([slerp(q_start,q_end,t_v.unsqueeze(0)) for t_v in t_interp]).squeeze(1)
v_interp= quat_rotate(q_slerp, torch.tensor([[1.,0.,0.]]).expand(100,-1))

fig = plt.figure(figsize=(15,4))
ax3d = fig.add_subplot(131, projection='3d')
ax3d.plot(v_interp[:,0].numpy(), v_interp[:,1].numpy(), v_interp[:,2].numpy(),
          'b-', lw=2, label='SLERP path')
u_sp,v_sp=np.mgrid[0:2*np.pi:30j,0:np.pi:15j]
ax3d.plot_surface(np.sin(v_sp)*np.cos(u_sp),np.sin(v_sp)*np.sin(u_sp),np.cos(v_sp),
                  alpha=0.1,color='cyan')
ax3d.scatter(*v_interp[0].numpy(),'go',s=80,zorder=5)
ax3d.scatter(*v_interp[-1].numpy(),'r^',s=80,zorder=5)
ax3d.set_title('SLERP on unit sphere'); ax3d.legend(fontsize=8)

ax_bm = fig.add_subplot(132)
for op_name,times_row in timing_results.items():   # loop: plot timing
    ax_bm.loglog(batch_sizes, times_row, 'o-', ms=5, lw=2, label=op_name)
ax_bm.set_xlabel('Batch size N'); ax_bm.set_ylabel('Time (ms)')
ax_bm.set_title('3D Torch op timing'); ax_bm.legend(fontsize=7); ax_bm.grid(True,alpha=0.3)

ax_tp = fig.add_subplot(133)
for op_name,times_row in timing_results.items():   # loop: throughput
    tp = [N_v/t*1e-6 for N_v,t in zip(batch_sizes,times_row)]
    ax_tp.semilogx(batch_sizes, tp,'o-',ms=5,lw=2,label=op_name)
ax_tp.set_xlabel('Batch size N'); ax_tp.set_ylabel('Throughput (Mops/s)')
ax_tp.set_title('Throughput'); ax_tp.legend(fontsize=7); ax_tp.grid(True,alpha=0.3)
plt.suptitle('🎯 Torch 3D: Quaternions · SLERP · Batch Timing',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §9 🏆 Gamer Science — ELO · Hitbox AABB · Ballistics · MMR

**ELO rating:** $E_A = \\frac{1}{1+10^{(R_B-R_A)/400}}$, $\\Delta R = K(S-E)$.  
**Hitbox AABB:** axis-aligned bounding box overlap in 3D — separating axis theorem.  
**Projectile ballistics:** drag $F_D=\\frac{1}{2}\\rho C_D A v^2$, gravity, wind.  
**MMR matchmaking:** minimize skill variance within match, maximize queue time satisfaction.

In [ ]:
# ── ELO rating system ─────────────────────────────────────────────
def elo_expected(Ra, Rb):
    return 1/(1+10**((Rb-Ra)/400))

def elo_update(Ra, Rb, score_a, K=32):
    """score_a: 1=win, 0.5=draw, 0=loss."""
    Ea = elo_expected(Ra, Rb)
    return Ra + K*(score_a - Ea), Rb + K*((1-score_a) - (1-Ea))

# Simulate ELO tournament
N_players = 16
ratings   = {f'P{i+1:02d}': 1000 + i*50 for i in range(N_players)}
true_skill= {p: 1000 + i*50 + np.random.randn()*100 for i,(p,_) in enumerate(ratings.items())}

history   = {p:[r] for p,r in ratings.items()}
N_rounds  = 200
players   = list(ratings.keys())

for rnd in range(N_rounds):         # loop: tournament rounds
    pa,pb = random.sample(players,2)
    # Win probability based on true skill
    p_win = elo_expected(true_skill[pa], true_skill[pb])
    score = 1 if random.random()<p_win else 0
    ratings[pa], ratings[pb] = elo_update(ratings[pa], ratings[pb], score)
    history[pa].append(ratings[pa])
    history[pb].append(ratings[pb])

# Rank correlation: ELO vs true skill
elo_rank  = sorted(players, key=lambda p: -ratings[p])
true_rank = sorted(players, key=lambda p: -true_skill[p])
from scipy.stats import spearmanr
elo_order  = [elo_rank.index(p)  for p in players]
true_order = [true_rank.index(p) for p in players]
rho, pval  = spearmanr(elo_order, true_order)
print(f'ELO after {N_rounds} rounds: Spearman ρ={rho:.3f} (ELO vs true skill, p={pval:.3f})')

# ── Hitbox: AABB collision detection ──────────────────────────────
class AABB:
    """Axis-Aligned Bounding Box."""
    def __init__(self, center, half_extents):
        self.c = np.array(center,dtype=float)
        self.h = np.array(half_extents,dtype=float)

    @property
    def lo(self): return self.c - self.h
    @property
    def hi(self): return self.c + self.h

    def intersects(self, other):
        return all(self.lo[i]<=other.hi[i] and other.lo[i]<=self.hi[i]
                   for i in range(3))

    def ray_intersect(self, ray_o, ray_d):
        """Slab method. Returns (t_min, t_max) or None."""
        t_min, t_max = -np.inf, np.inf
        for i in range(3):           # loop: slab test per axis
            if abs(ray_d[i]) < 1e-10:
                if ray_o[i]<self.lo[i] or ray_o[i]>self.hi[i]: return None
            else:
                t1=(self.lo[i]-ray_o[i])/ray_d[i]
                t2=(self.hi[i]-ray_o[i])/ray_d[i]
                t_min=max(t_min,min(t1,t2)); t_max=min(t_max,max(t1,t2))
        return (t_min,t_max) if t_max>=max(t_min,0) else None

# Test 1000 collision pairs
n_hits=0; n_tests=1000
for _ in range(n_tests):            # loop: collision tests
    cA=np.random.randn(3)*5; hA=np.random.rand(3)+0.5
    cB=np.random.randn(3)*5; hB=np.random.rand(3)+0.5
    if AABB(cA,hA).intersects(AABB(cB,hB)): n_hits+=1
print(f'AABB: {n_hits}/{n_tests} collisions ({100*n_hits/n_tests:.1f}%)')

# ── Ballistics with drag ───────────────────────────────────────────
from scipy.integrate import solve_ivp
g_v=9.81; rho_air=1.225; Cd=0.47; A_proj=np.pi*(0.05)**2; m_proj=0.5
wind=np.array([5.0,0.0,0.0])  # 5 m/s headwind

def ballistics_ode(t,y, use_drag=True, use_wind=True):
    x,z,vx,vz = y
    v_rel = np.array([vx,vz]) - (np.array([wind[0],0.]) if use_wind else 0)
    v_mag = np.linalg.norm(v_rel)+1e-10
    drag  = -0.5*rho_air*Cd*A_proj*v_mag/m_proj * v_rel if use_drag else np.zeros(2)
    return [vx, vz, drag[0], -g_v+drag[1]]

launch_angles = [20,30,45,60,70]
v0 = 50.0  # m/s muzzle velocity
scenarios = [('Vacuum',False,False),('Drag only',True,False),('Drag+Wind',True,True)]

fig, axes = plt.subplots(1, 3, figsize=(15,4))

for ax_b, (label,use_d,use_w) in zip(axes, scenarios):  # loop: scenarios
    for ang in launch_angles:                            # loop: launch angles
        th = np.radians(ang)
        y0 = [0,0, v0*np.cos(th), v0*np.sin(th)]
        sol= solve_ivp(lambda t,y: ballistics_ode(t,y,use_d,use_w),
                       [0,20], y0, events=[lambda t,y,*a: y[1]],
                       dense_output=True, max_step=0.05)
        t_land = sol.t[-1]
        t_plot = np.linspace(0,t_land,200)
        traj   = sol.sol(t_plot)
        ax_b.plot(traj[0], traj[1], lw=1.5, label=f'{ang}°')
    ax_b.set_xlabel('Range (m)'); ax_b.set_ylabel('Height (m)')
    ax_b.set_title(label); ax_b.legend(fontsize=7); ax_b.grid(True,alpha=0.3)
    ax_b.set_ylim(bottom=0)

plt.suptitle('🏆 Gamer Science: Ballistics (v₀=50m/s)',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

# ── MMR Matchmaking ────────────────────────────────────────────────
N_plyr = 500; N_slots=5; N_matches=1000
mmr_pool = np.random.normal(2500, 400, N_plyr).clip(0,5000)
queue_times = np.zeros(N_plyr)  # time in queue
match_skills = []; wait_times=[]

for match in range(N_matches):   # loop: matchmaking
    queue_times += 1
    # Sort by MMR; match closest 10 players
    sorted_idx = np.argsort(mmr_pool)
    # Pick a random window of 10 from sorted list
    start = random.randint(0, len(sorted_idx)-10)
    selected = sorted_idx[start:start+10]
    skill_var = np.std(mmr_pool[selected])
    match_skills.append(skill_var)
    wait_times.extend(queue_times[selected].tolist())
    queue_times[selected] = 0

fig, axes2 = plt.subplots(1,3,figsize=(15,4))
# ELO history
for p in list(history.keys())[:6]:   # loop: plot top 6 players
    axes2[0].plot(history[p], lw=1.5, label=p, alpha=0.8)
axes2[0].set_xlabel('Game #'); axes2[0].set_ylabel('ELO rating')
axes2[0].set_title('ELO convergence'); axes2[0].legend(fontsize=7); axes2[0].grid(True,alpha=0.3)

axes2[1].hist(match_skills, bins=40, color='steelblue', edgecolor='k', lw=0.3)
axes2[1].axvline(np.mean(match_skills),ls='--',color='r',lw=2,label=f'Mean={np.mean(match_skills):.0f}')
axes2[1].set_xlabel('Skill std dev in match'); axes2[1].set_ylabel('Count')
axes2[1].set_title('Match quality (lower=better)'); axes2[1].legend(); axes2[1].grid(True,alpha=0.3)

axes2[2].hist(wait_times, bins=40, color='orange', edgecolor='k', lw=0.3)
axes2[2].axvline(np.mean(wait_times),ls='--',color='r',lw=2,label=f'Mean={np.mean(wait_times):.1f}')
axes2[2].set_xlabel('Queue time (ticks)'); axes2[2].set_ylabel('Count')
axes2[2].set_title('MMR matchmaking queue time'); axes2[2].legend(); axes2[2].grid(True,alpha=0.3)
plt.suptitle('🏆 Gamer Science: ELO · MMR Matchmaking',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

print('\n✦ GAMER SCIENCE COMPLETE ✦')
print(f'  ELO Spearman ρ = {rho:.3f}')
print(f'  AABB hit rate  = {100*n_hits/n_tests:.1f}%')
print(f'  Avg match σ    = {np.mean(match_skills):.1f} MMR')
print(f'  Avg queue time = {np.mean(wait_times):.1f} ticks')